In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import torch
import torch.nn as nn

import os
import glob

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Hardware Acceleration: Apple Silicon MPS (Metal Performance Shaders) enabled.")
else:
    device = torch.device("cpu")
    print("Hardware Acceleration: MPS not found. Defaulting to CPU.")

Hardware Acceleration: Apple Silicon MPS (Metal Performance Shaders) enabled.


# Deep Learning for UK Electricity Demand Forecasting During Extreme Summer Heatwaves

### 1. Introduction

#### 1.1 Background
Extreme heatwaves place operational stress on the UK electricity grid. Accurate short-term National Demand (ND) forecasting during high-temperature events is critical for the National Energy System Operator (NESO) to maintain grid stability and balance resources.

#### 1.2 Problem Statement
Standard forecasting models trained on typical seasonal profiles may struggle with non-linear demand dynamics during severe weather anomalies, such as the July 2022 UK heatwave (>40°C). Peak forecasting errors during extreme events pose operational and financial risks to grid management.

#### 1.3 Research Aim
This study evaluates whether a sequence-based Long Short-Term Memory (LSTM) network incorporating historical demand and Met Office weather observations improves short-term demand forecasting during heatwaves compared to an XGBoost benchmark.

#### 1.4 Research Objectives
* **Data Pipeline:** Combine half-hourly UK National Demand data with MIDAS weather observations across seven key regions (2019–2025).
* **Feature Engineering:** Construct temporal, solar, and thermal metrics, including rolling temperatures and heat persistence.
* **Baseline Model:** Implement an XGBoost Regressor as a non-sequential benchmark, chosen for its strong performance on tabular time-series data.
* **Deep Learning Model:** Train a sequence-to-value LSTM utilizing a 24-hour (48-timestep) lookback window.
* **Evaluation:** Compare out-of-sample performance, isolating the July 2022 heatwave, using MAE, RMSE, $R^2$, and peak forecasting error.

#### 1.5 Research Question
Does a sequence-based deep-learning model using historical demand and weather data improve short-term electricity demand forecasting during extreme heatwaves compared to a standard tree-based benchmark?

### 2. Data Collection & Preprocessing

#### 2.1 Hardware Configuration & Environment
The experimental pipeline is executed locally utilizing Apple Silicon (M4) optimization. Deep learning models are built in PyTorch and routed through the Metal Performance Shaders (MPS) backend. The machine learning benchmark utilizes the `xgboost` framework, while standard temporal alignments and array manipulations are handled via `pandas` and `numpy`.

#### 2.2 Dataset Overview
This study relies on two primary data streams spanning 2019 to 2025:
* **Electricity Demand Data:** Half-hourly grid data featuring the target variable, National Demand (ND) in Megawatts (MW), alongside Embedded Solar Generation estimates.
* **Meteorological Data:** Hourly surface observations sourced from the MIDAS Open dataset. To capture regional variations in grid load, data is aggregated from seven key UK weather stations: Heathrow, Birmingham, Manchester, Leeds, Bristol, Cardiff, and Edinburgh. The primary variables extracted are air temperature and relative humidity.

#### 2.3 Temporal Alignment & Interpolation
A critical preprocessing step involves synchronizing the temporal resolution of the distinct datasets. The UK electricity market operates on 30-minute settlement periods, whereas the MIDAS weather observations are recorded hourly. To resolve this discrepancy without discarding high-resolution demand data, the meteorological data is resampled to a 30-minute frequency using time-based linear interpolation. Missing values are forward and backward filled at the dataset boundaries, resulting in a continuous, fully aligned dataframe suitable for engineering lagged sequences and heatwave indicators.

In [15]:
# 1. Define Directory Paths

demand_dir = "data/demand_data"
weather_dir = "data/uk_hourly_weather_data"

# 2. Helper Function to Skip Weather Metadata

def find_header_row(filepath, keyword="ob_time"):
    """
    Scans the weather CSV to find the exact header row starting with 'ob_time'
    and skips the metadata rows.
    """
    try:
        with open(filepath, 'r') as file:
            for i, line in enumerate(file):
                if line.startswith(keyword):
                    return i
        return 0 
    except FileNotFoundError:
        print(f"Error: Could not find {filepath}")
        return 0

# 3. Batch Load Demand Data

demand_filepaths = glob.glob(os.path.join(demand_dir, "demanddata_*.csv"))

demand_data_list = []
for filepath in demand_filepaths:
    temp_demand = pd.read_csv(filepath)
    
    temp_demand['Datetime'] = pd.to_datetime(temp_demand['SETTLEMENT_DATE'], format='mixed') + \
                              pd.to_timedelta((temp_demand['SETTLEMENT_PERIOD'] - 1) * 30, unit='m')
    temp_demand.set_index('Datetime', inplace=True)
    
    temp_demand = temp_demand[['ND', 'EMBEDDED_SOLAR_GENERATION']]
    demand_data_list.append(temp_demand)

combined_demand_data = pd.concat(demand_data_list)
combined_demand_data.sort_index(inplace=True)

combined_demand_data = combined_demand_data[~combined_demand_data.index.duplicated(keep='first')]

combined_demand_data = combined_demand_data.resample('30min').asfreq()
combined_demand_data['ND'] = combined_demand_data['ND'].interpolate(method='time')
combined_demand_data['EMBEDDED_SOLAR_GENERATION'] = combined_demand_data['EMBEDDED_SOLAR_GENERATION'].fillna(0)

# 4. Batch Load Weather Data

weather_filepaths = glob.glob(os.path.join(weather_dir, "*.csv"))

weather_data_list = []
for filepath in weather_filepaths:
    header_idx = find_header_row(filepath, keyword="ob_time")
    
    temp_weather = pd.read_csv(filepath, skiprows=header_idx, engine='python', on_bad_lines='skip')
    
    if 'ob_time' in temp_weather.columns:
        temp_weather['ob_time'] = pd.to_datetime(temp_weather['ob_time'], format='mixed', errors='coerce')
        temp_weather.dropna(subset=['ob_time'], inplace=True)
        
        temp_weather = temp_weather[['ob_time', 'air_temperature', 'rltv_hum']]
        
        temp_weather['air_temperature'] = pd.to_numeric(temp_weather['air_temperature'], errors='coerce')
        temp_weather['rltv_hum'] = pd.to_numeric(temp_weather['rltv_hum'], errors='coerce')
        
        weather_data_list.append(temp_weather)

raw_weather_data = pd.concat(weather_data_list)

aggregated_weather_data = raw_weather_data.groupby('ob_time').mean()

aligned_weather_data = aggregated_weather_data.resample('30min').interpolate(method='time')
aligned_weather_data.ffill(inplace=True)
aligned_weather_data.bfill(inplace=True)

# 5. Merge Datasets

merged_dataset = combined_demand_data.join(aligned_weather_data, how='inner')

print(f"Final dataset shape: {merged_dataset.shape}")
print(f"Date range: {merged_dataset.index.min()} to {merged_dataset.index.max()}")
merged_dataset.rename(columns={
    'ND': 'national_demand_mw',
    'EMBEDDED_SOLAR_GENERATION': 'embedded_solar_generation_mw',
    'air_temperature': 'air_temperature_deg_c',
    'rltv_hum': 'relative_humidity_pct'
}, inplace=True)

display(merged_dataset.head())

Final dataset shape: (122735, 4)
Date range: 2019-01-01 00:00:00 to 2025-12-31 23:00:00


,national_demand_mw,embedded_solar_generation_mw,air_temperature_deg_c,relative_humidity_pct
2019-01-01 00:00:00,23808.0,0.0,9.214286,79.785714
2019-01-01 00:30:00,24402.0,0.0,9.207143,80.264286
2019-01-01 01:00:00,24147.0,0.0,9.200000,80.742857
2019-01-01 01:30:00,23197.0,0.0,9.057143,80.564286
2019-01-01 02:00:00,22316.0,0.0,8.914286,80.385714
